# 03 · Evaluation and Testing

This notebook measures retrieval and generation quality across a fixed question set.

Metrics computed:
| Metric | Description |
|--------|-------------|
| **Hit Rate @ k** | Fraction of questions where the correct chunk is in the top-k results |
| **MRR** | Mean Reciprocal Rank — rewards higher-ranked correct results |
| **Faithfulness** | Does the answer only use the provided context? |
| **Answer Relevance** | Is the answer relevant to the question? |

**Prerequisites**: `02_rag_query_pipeline.ipynb` must have been run (`vector_store` and `qa_chain` in scope).

In [ ]:
# Evaluation question set — ground-truth answers drawn from source PDFs
EVAL_SET = [
    {
        "question": "What is PropTech?",
        "expected_keywords": ["property technology", "real estate", "digital"],
    },
    {
        "question": "What are the main challenges in real estate technology adoption?",
        "expected_keywords": ["data", "integration", "legacy", "compliance"],
    },
    {
        "question": "How do AI tools improve the sales process for real estate agents?",
        "expected_keywords": ["automation", "lead", "efficiency", "time"],
    },
    {
        "question": "What role does data play in PropTech decision-making?",
        "expected_keywords": ["data", "analytics", "insights", "decision"],
    },
    {
        "question": "What is the future of AI in real estate?",
        "expected_keywords": ["AI", "automation", "future", "market"],
    },
]

print(f"Evaluation set: {len(EVAL_SET)} questions")

## Retrieval Evaluation

For each question, we run similarity search and measure whether the returned chunks
contain the expected keywords (proxy for ground-truth relevance without manual labelling).

In [ ]:
def keyword_hit(chunks, keywords: list[str]) -> tuple[bool, int | None]:
    """Return (hit, rank) where rank is 1-indexed position of first matching chunk."""
    for i, chunk in enumerate(chunks):
        text = chunk.page_content.lower()
        if any(kw.lower() in text for kw in keywords):
            return True, i + 1
    return False, None


K_VALUES = [1, 3, 5]
results = []

for item in EVAL_SET:
    q = item["question"]
    expected = item["expected_keywords"]

    chunks = vector_store.similarity_search(q, k=max(K_VALUES))
    hit, rank = keyword_hit(chunks, expected)

    row = {"question": q, "hit": hit, "rank": rank}
    for k in K_VALUES:
        row[f"hit@{k}"] = hit and rank is not None and rank <= k
    row["rr"] = 1 / rank if rank else 0.0
    results.append(row)
    print(f"  Q: {q[:60]}")
    print(f"     Hit: {hit} | Rank: {rank} | RR: {row['rr']:.3f}")

print()

In [ ]:
import statistics

n = len(results)
mrr = statistics.mean(r["rr"] for r in results)

print("=" * 50)
print("RETRIEVAL EVALUATION SUMMARY")
print("=" * 50)
for k in K_VALUES:
    hit_rate = sum(1 for r in results if r[f"hit@{k}"]) / n
    print(f"  Hit Rate @ {k}: {hit_rate:.2f}")
print(f"  MRR         : {mrr:.3f}")

## Generation Quality (Faithfulness Proxy)

A faithful answer only uses information from the retrieved context.
We measure this heuristically: if the answer contains phrases not present in any retrieved chunk, it may be hallucinating.

For production, replace this with [RAGAS](https://docs.ragas.io/) or a dedicated judge LLM.

In [ ]:
def faithfulness_score(answer: str, chunks: list) -> float:
    """Fraction of answer 4-grams that appear in at least one retrieved chunk."""
    context = " ".join(c.page_content.lower() for c in chunks)
    words = answer.lower().split()
    if len(words) < 4:
        return 1.0  # too short to measure
    ngrams = [" ".join(words[i:i+4]) for i in range(len(words) - 3)]
    hits = sum(1 for ng in ngrams if ng in context)
    return hits / len(ngrams)


gen_results = []
for item in EVAL_SET:
    q = item["question"]
    chunks = vector_store.similarity_search(q, k=4)
    result = qa_chain.invoke({"input": q})
    answer = result["answer"]
    faith = faithfulness_score(answer, chunks)
    gen_results.append({"question": q, "answer": answer, "faithfulness": faith})
    print(f"Q: {q[:60]}")
    print(f"   Faithfulness: {faith:.2f}")
    print(f"   Answer snippet: {answer[:120]}...\n")

In [ ]:
avg_faith = statistics.mean(r["faithfulness"] for r in gen_results)

print("=" * 50)
print("GENERATION EVALUATION SUMMARY")
print("=" * 50)
print(f"  Avg Faithfulness: {avg_faith:.2f}")
print()
print("Full results table:")
print(f"{'Question':<55} {'Faithfulness':>12}")
print("-" * 70)
for r in gen_results:
    print(f"{r['question'][:55]:<55} {r['faithfulness']:>12.2f}")

## Summary

| Metric | Value |
|--------|-------|
| Hit Rate @ 1 | see output above |
| Hit Rate @ 3 | see output above |
| MRR | see output above |
| Avg Faithfulness | see output above |

**Next steps for improving retrieval accuracy**:
- Add Cohere Rerank as a post-retrieval step (see RAGAS docs)
- Increase chunk overlap for documents with dense cross-references
- Add metadata filters to restrict search to specific document types